In [138]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [139]:
class KNNRegression:
    def __init__ (self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = []
        for x in X:
            # Calculate distances to all training points
            distances = np.linalg.norm(self.X_train - x, axis=1)
            
            # Get indices of k nearest neighbors
            k_indices = np.argpartition(distances, self.k)[:self.k]

            # Calculate mean value of k nearest neighbors
            k_nearest_values = self.y_train[k_indices]
            mean_value = np.mean(k_nearest_values)

            predictions.append(mean_value)
        return np.array(predictions)

In [140]:
def scaled_data(X):
    X_scaled = (X - X.min())/(X.max() - X.min())
    return X_scaled

In [141]:
data = pd.read_csv('50_Startups.csv')
data.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [142]:
data = pd.get_dummies(data, columns=['State'], dtype=float)
data.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_California,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,0.0,0.0,1.0
1,162597.70,151377.59,443898.53,191792.06,1.0,0.0,0.0
2,153441.51,101145.55,407934.54,191050.39,0.0,1.0,0.0
3,144372.41,118671.85,383199.62,182901.99,0.0,0.0,1.0
4,142107.34,91391.77,366168.42,166187.94,0.0,1.0,0.0


In [143]:
X = data[['R&D Spend', 'Administration', 'Marketing Spend', 'State_California', 'State_Florida', 'State_New York']].values
y = data['Profit'].values

In [144]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=36)

scaled_data(X_train)
scaled_data(X_test)

print(X_train.shape)
print(X_test.shape)

(40, 6)
(10, 6)


In [145]:
def evaluate(model, X, y):
    y_pred = model.predict(X)
    
    mse = (1 / len(X)) * np.sum((y_pred - y)**2)
    rmse = np.sqrt(mse)

    mae = (1 / len(X)) * np.sum(abs(y - y_pred))
    
    y_mean = np.sum(y) / len(y)
    SS_res = np.sum((y - y_pred)**2)
    SS_tot = np.sum((y - y_mean)**2)
    
    r2 = (1 - SS_res/SS_tot)
    return mse, rmse, mae, r2

In [146]:
model = KNNRegression(k=5)
model.fit(X_train, y_train)

In [147]:
mse, rmse, mae, r2 = evaluate(model, X_test, y_test)

print(f'rmse: {rmse}')
print(f'mae: {mae}')
print(f'r2: {r2}')

rmse: 11023.55168427271
mae: 8574.621799999997
r2: 0.8761873963231557


In [148]:
import numpy as np
from sklearn.neighbors import KNeighborsRegressor


sk_model = KNeighborsRegressor(n_neighbors=3)
sk_model.fit(X_train, y_train)
sk_y_pred = sk_model.predict(X_test)

 # Evaluation
print(f'RMSE (Sklearn): {np.sqrt(np.mean(( y_test - sk_y_pred ) ** 2))}')

RMSE (Sklearn): 9580.88525301267


In [149]:
class KNNClassification:
    def __init__(self, k=5):
        self.k = k
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y.reshape(-1).astype(int)

    def predict(self, X):
        predictions = []
        for x in X:
            distances = np.sqrt(np.sum((self.X_train - x)**2, axis=1))
            
            k_indices = np.argpartition(distances, self.k - 1)[:self.k]
            k_nearest_labels = self.y_train[k_indices]

            most_common = np.bincount(k_nearest_labels).argmax()
            predictions.append(most_common)

        return np.array(predictions)

In [150]:
data = pd.read_csv('Iris.csv')
data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [151]:
data['Species'] = data['Species'].str.strip().str.lower()

data['Species'] = data['Species'].map({
    'iris-setosa': 0,
    'iris-versicolor': 1,
    'iris-virginica': 2
})
data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,0
1,2,4.9,3.0,1.4,0.2,0
2,3,4.7,3.2,1.3,0.2,0
3,4,4.6,3.1,1.5,0.2,0
4,5,5.0,3.6,1.4,0.2,0


In [152]:
X = data[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']].values
y = data['Species'].values 

print(X.shape)

(150, 4)


In [153]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

X_train = scaled_data(X_train)
X_test = scaled_data(X_test)

print(X_test.shape)

(30, 4)


In [154]:
model = KNNClassification(k=5)
model.fit(X_train, y_train)

In [155]:
def accuracy_classification(model, X, y):
    y_pred = model.predict(X)
    y = y.reshape(-1)
    y_pred = y_pred.reshape(-1)
    return np.mean(y == y_pred)


In [156]:
acc = accuracy_classification(model, X_test, y_test)
print(acc)

1.0


In [157]:
from sklearn . neighbors import KNeighborsClassifier
from sklearn . metrics import accuracy_score

# Scikit - learn model
sk_model = KNeighborsClassifier(n_neighbors =5)
sk_model.fit(X_train, y_train)
sk_y_pred = sk_model.predict(X_test)

# Evaluation
print (f'Accuracy (Sklearn): {accuracy_score(y_test, sk_y_pred)}' )

Accuracy (Sklearn): 1.0
